# Mergers using Non-Homologous TARDIS Workflow

In [ ]:
from numba import config as nbconfig

nbconfig.DISABLE_JIT = True

In [ ]:
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy import units as u
from plot_manga import plot_manga_profiles

from tardis.io.configuration.config_reader import Configuration
from tardis.workflows.nonhomologous_tardis_workflow import (
    NonhomologousTARDISWorkflow,
)

In [ ]:
# Load in the MANGA data
time_explosion = 400 # days
fname = f"bins_{time_explosion}days.dat"
nangles = 19
nradii = 599
angle_bin = 9

index = pd.MultiIndex.from_product([range(nangles), range(nradii)], names=['angle bin', 'radius bin'])
data = pd.DataFrame(np.loadtxt(fname, comments=['#', '\n']), columns=["r", "dens", "temp", "v_r"], index=index)

data_1d = data.loc[angle_bin]

#data_1d_cut = data_1d.loc[np.logical_and(data_1d['r'] > 3.5e13, data_1d['r'] < 1e15)]
data_1d_cut = data_1d.loc[np.logical_and(data_1d['r'] > 3.9e13, data_1d['r'] < 1e15)]
nshells = len(data_1d_cut) - 1

# Floor the temperature at 1e3 K
data_1d_cut.loc[data_1d_cut['temp'] < 1e3, "temp"] = 1000.01

# TEMPORARY: Artificially reduce density
#data_1d_cut['dens'] *= 1e-2

# Change selected_bin to highlight a different line of sight
fig = plot_manga_profiles(data, nangles, selected_bin=angle_bin, day=time_explosion)
axes = fig.axes

# Set label for the MANGA data
axes[5].lines[0].set_label('MANGA 1D data for highlighted bin')

# Overplot cut data
axes[3].plot(data_1d_cut['r'], data_1d_cut['v_r'], c="C1", lw=5, alpha=0.3)
axes[4].plot(data_1d_cut['r'], data_1d_cut['dens'], c="C1", lw=5, alpha=0.3)
axes[5].plot(data_1d_cut['r'], data_1d_cut['temp'], c="C1", lw=5, alpha=0.3, label='Subset of data used for TARDIS sim')

# Create homologous data to compare with
data_homologous = data_1d_cut.copy(deep=True)
data_homologous['v_r'] = (data_homologous['r'] / (time_explosion * u.day.to(u.s)))
#axes[3].plot(data_homologous['r'], data_homologous['v_r'], c="C2", lw=3, alpha=0.3, linestyle='--')

# Plot fiducial tardis example profiles for comparison
config = Configuration.from_yaml('../../tardis_example.yml')
workflow = NonhomologousTARDISWorkflow(config)
v_fiducial = np.linspace(config.model.structure.velocity.start, config.model.structure.velocity.stop, config.model.structure.velocity.num)
dens_fiducial = workflow.simulation_state.composition.density
temp_fiducial = workflow.simulation_state.t_radiative
r_fiducial = (v_fiducial*time_explosion*u.day).to(u.cm)
axes[3].plot(r_fiducial, v_fiducial.to(u.cm/u.s), c='C3')
axes[4].plot(r_fiducial, dens_fiducial, c='C3')
axes[5].plot(r_fiducial, temp_fiducial, c='C3', label='tardis_example.yml (for reference)')
axes[3].set_xscale('log')
axes[4].set_xscale('log')
axes[5].set_xscale('log')
axes[5].legend(frameon=False)
plt.show()

In [ ]:

v_km_s = (data_1d_cut['v_r'].values * u.cm / u.s).to(u.km / u.s).value
r_cm   = data_1d_cut['r'].values
rho    = data_1d_cut['dens'].values
t_rad  = data_1d_cut['temp'].values

csvy_yaml_header = textwrap.dedent(f"""\
    ---
    name: manga_merger_{time_explosion}days_bin{angle_bin}
    model_density_time_0: {time_explosion} day
    model_isotope_time_0: nan s
    description: MANGA merger non-homologous model at {time_explosion} days, angle bin {angle_bin}
    tardis_model_config_version: v1.0
    datatype:
      fields:
        - name: velocity
          unit: km/s
          desc: velocities of shell boundaries (inner-to-outer)
        - name: radius
          unit: cm
          desc: actual radial positions of shell boundaries (non-homologous)
        - name: density
          unit: g/cm^3
          desc: shell density
        - name: t_rad
          unit: K
          desc: initial radiative temperature
        - name: H
          desc: fractional H abundance (pure hydrogen)
    v_inner_boundary: {v_km_s[0]:.6f} km/s
    v_outer_boundary: {v_km_s[-1]:.6f} km/s
    ---
""")

csv_lines = ["velocity,radius,density,t_rad,H"]
for v, r, rho_val, t_val in zip(v_km_s, r_cm, rho, t_rad):
    csv_lines.append(f"{v:.6f},{r:.6e},{rho_val:.6e},{t_val:.4f},1.0")

csvy_content = csvy_yaml_header + "\n".join(csv_lines) + "\n"

csvy_path = Path("manga_merger.csvy")
csvy_path.write_text(csvy_content)
print(f"Wrote CSVY to {csvy_path.resolve()}")
print(csvy_content[:700])


In [ ]:
import yaml

# Start with example yaml config
base_cfg_dict = yaml.safe_load(Path('../../tardis_example.yml').read_text())

# Force cmfgen atom data instead of chianti - potentially bad H data in default file
base_cfg_dict['atom_data'] = "./kurucz_cd23_cmfgen_H_He.h5"

# Remove the model block since structure and abundances come from the CSVY file
base_cfg_dict.pop('model', None)
base_cfg_dict['csvy_model'] = str(csvy_path)

base_cfg_dict['supernova']['time_explosion'] = f'{time_explosion} day'
base_cfg_dict.setdefault('montecarlo', {})['enable_full_relativity'] = False

# Specific to my laptop:
base_cfg_dict['montecarlo']['nthreads'] = 22

# Overwrite spectrum bins
#base_cfg_dict['spectrum']['start'] = '10 angstrom'
#base_cfg_dict['spectrum']['stop'] = '2000 angstrom'

# Overwrite packet count
base_cfg_dict['montecarlo']['last_no_of_packets'] = 10000.0
base_cfg_dict['montecarlo']['no_of_packets'] = 1000.0

# T_inner - set luminosity to be ~100x smaller than a SN for a luminous red nova
# Set damping constant to zero so no T_inner iteration is done - leave the temperature
# alone since we're getting it from the merger model directly
base_cfg_dict['supernova']['luminosity_requested'] = "7.44 log_lsun"
base_cfg_dict['montecarlo']['convergence_strategy']['t_inner']['damping_constant'] = 0.0

# Write out the config dict to yaml and load it into a Configuration object
config_yaml_path = Path("manga_merger_config.yml")
config_yaml_path.write_text(yaml.dump(base_cfg_dict, default_flow_style=False))
print(f"Wrote config to {config_yaml_path.resolve()}")

config = Configuration.from_yaml(str(config_yaml_path))

In [ ]:
workflow = NonhomologousTARDISWorkflow(config, csvy=True)

# Find the Photosphere (tau = 2/3)

Using pure H, calculate the location of the tau=2/3 surface in the MANGA models to make sure we're not too optically thick.

Uses the Rosseland mean opacity (electron scattering + line optical depth), integrated from the outside in, following the same method as in the `v_inner_solver_workflow`.

## Note:

The merger model temperatures can get pretty low, and they're floored at T=1000 K. Hydrogen is fully neutral here, so there won't be any electron number density, meaning the optical depth from electron scattering will be zero for these regions.

In [ ]:
from tardis.workflows.util import get_tau_integ

opacity_states = workflow.solve_opacity()
tau_integ = get_tau_integ(
    workflow.plasma_solver,
    opacity_states['opacity_state'],
    workflow.simulation_state,
)
tau_integ['rosseland'].to(1)

## Just electron scattering opacity

In [ ]:
from astropy import constants as const

# Electron scattering only: kappa_es = n_e * sigma_T
n_e = workflow.plasma_solver.electron_densities.values * u.cm**-3
kappa_es = (n_e * const.sigma_T).to(u.cm**-1)

dr = (workflow.simulation_state.geometry.r_outer - workflow.simulation_state.geometry.r_inner).to(u.cm)
dtau_es = (kappa_es * dr).to(u.dimensionless_unscaled)

# Integrate from outside in (cumsum reversed)
tau_es_integrated = np.cumsum(dtau_es[::-1])[::-1]
print(tau_es_integrated)


## Just a single line

In [ ]:
tau_sobolev = opacity_states['opacity_state'].tau_sobolev

# Pick the line with the largest total tau_sobolev (summed over shells)
line_pos = tau_sobolev.sum(axis=1).values.argmax()
line_id = tau_sobolev.index[line_pos]
line_info = workflow.plasma_solver.atomic_data.lines.iloc[line_pos]

wavelength = (line_info['nu']*u.Hz).to(u.AA, equivalencies=u.spectral())
print(f"Strongest line: {[int(i) for i in line_id]}, wavelength={wavelength:.1f} Å")

tau_sobolev_line = tau_sobolev.iloc[line_pos].values
tau_line_integrated = np.cumsum(tau_sobolev_line[::-1])[::-1]
tau_line_integrated

In [ ]:
plt.plot(r_cm[1:], tau_integ['rosseland'].to(1), label='Rosseland Mean')
plt.plot(r_cm[1:], tau_es_integrated, label='Electron Scattering')
#plt.plot(r_cm[1:], tau_line_integrated, label='Strongest Line')
plt.ylabel(r"$\tau$")
plt.xlabel("r (cm)")
plt.axhline(2./3, c='r', ls='--', label=r'$\tau = 2/3$')
plt.loglog()
plt.legend(frameon=False)

# To do next:

 - Take a typical packet starting frequency and see what maximum red or blue shift it could experience while traveling through the velocity profile from the model
 - Use that to estimate the amount of line opacity it's traveling through - answer the question: "are the packets getting stuck?"

In [ ]:
workflow.run()

In [ ]:
workflow.simulation_state.t_inner

In [ ]:
spectrum = workflow.spectrum_solver.spectrum_real_packets
spectrum_virtual = workflow.spectrum_solver.spectrum_virtual_packets
spectrum_integrated = workflow.spectrum_solver.spectrum_integrated

In [ ]:
spectrum = workflow.spectrum_solver.spectrum_real_packets
spectrum_virtual = workflow.spectrum_solver.spectrum_virtual_packets
spectrum_integrated = workflow.spectrum_solver.spectrum_integrated
hom_spectrum = hom_workflow.spectrum_solver.spectrum_real_packets
hom_spectrum_virtual = hom_workflow.spectrum_solver.spectrum_virtual_packets
hom_spectrum_integrated = hom_workflow.spectrum_solver.spectrum_integrated

In [ ]:
%matplotlib inline
plt.figure(figsize=(10, 6.5))

spectrum.plot(label="Normal packets, non-homology", c='C0', alpha=0.1)
#hom_spectrum.plot(label="Normal packets, homology", c='C1', alpha=0.1)

spectrum_virtual.plot(linewidth=3, c='w', alpha=1.0)
#hom_spectrum_virtual.plot(linewidth=4, c='w', alpha=1.0)
spectrum_virtual.plot(label="Virtual packets, non-homology", c='C0', alpha=0.35)
#hom_spectrum_virtual.plot(label="Virtual packets, homology", c='C1', alpha=0.35)

#spectrum_integrated.plot(linewidth=3, c='w', alpha=1.0)
#spectrum_integrated.plot(label="Formal Integral, non-homology", c='C0', alpha=1.0)
#hom_spectrum_integrated.plot(label="Formal Integral, homology", c='C1', alpha=1.0, linestyle='--')


plt.xlim(500, 20000)
plt.title("TARDIS example model spectrum")
plt.xlabel(r"Wavelength [$\AA$]")
plt.ylabel(r"Luminosity density [erg/s/$\AA$]")
plt.legend()
plt.show()